In [1]:
import os
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image

# Paths
BASE_INPUT = Path("..") / "input" / "cassava-leaf-disease-classification"
TRAIN_CSV = BASE_INPUT / "train.csv"
TRAIN_IMAGES_DIR = BASE_INPUT / "train_images"
TEST_IMAGES_DIR = BASE_INPUT / "test_images"
SAMPLE_SUBMIT = BASE_INPUT / "sample_submission.csv"
SUBMISSION_PATH = Path("./submission.csv")



In [2]:
# Load training metadata
train_df = pd.read_csv(TRAIN_CSV)
label_names = sorted(train_df["label"].unique())
num_classes = len(label_names)

# Simple mapping label -> idx (labels are already ints 0‑4, but ensure contiguous)
label_to_idx = {int(l): i for i, l in enumerate(label_names)}
idx_to_label = {i: int(l) for l, i in label_to_idx.items()}

print(f"Found {len(train_df)} training samples, {num_classes} classes.")




Found 18721 training samples, 5 classes.


In [3]:
class CassavaDataset(Dataset):
    def __init__(self, df, images_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.images_dir / row["image_id"]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = label_to_idx[int(row["label"])]
        return image, label


# Standard ImageNet transforms resized to 224×224
train_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
val_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)



In [4]:
# Create dataset and split (90% train, 10% val)
full_dataset = CassavaDataset(train_df, TRAIN_IMAGES_DIR, transform=train_transform)
val_size = int(0.1 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)
# Use validation transform for the val split
val_dataset.dataset.transform = val_transform

batch_size = 64
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)



In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


def evaluate(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, targets in loader:
            imgs = imgs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)
    return correct / total if total > 0 else 0.0


epochs = 3
best_val_acc = 0.0
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for imgs, targets in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / train_size
    val_acc = evaluate(val_loader)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = model.state_dict()
    print(f"Epoch {epoch}/{epochs} - Loss: {epoch_loss:.4f} - Val Acc: {val_acc:.4f}")

# Load best model
model.load_state_dict(best_state)
print(f"Best validation accuracy: {best_val_acc:.4f}")



Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:01<00:00, 45.7MB/s]


Epoch 1/3 - Loss: 0.6275 - Val Acc: 0.8184
Epoch 2/3 - Loss: 0.2948 - Val Acc: 0.7842
Epoch 3/3 - Loss: 0.1005 - Val Acc: 0.8007
Best validation accuracy: 0.8184


In [6]:
# Prepare test dataframe (use sample submission if present for ordering)
if SAMPLE_SUBMIT.exists():
    test_df = pd.read_csv(SAMPLE_SUBMIT)
else:
    test_images = [f.name for f in TEST_IMAGES_DIR.glob("*.jpg")]
    test_df = pd.DataFrame({"image_id": test_images})

# Predict
model.eval()
test_transform = val_transform  # same as validation
pred_labels = []

with torch.no_grad():
    for idx in range(0, len(test_df), batch_size):
        batch_ids = test_df["image_id"].iloc[idx : idx + batch_size]
        batch_imgs = []
        for img_name in batch_ids:
            img_path = TEST_IMAGES_DIR / img_name
            image = Image.open(img_path).convert("RGB")
            batch_imgs.append(test_transform(image))
        batch_tensor = torch.stack(batch_imgs).to(device)
        outputs = model(batch_tensor)
        _, preds = torch.max(outputs, 1)
        pred_labels.extend([idx_to_label[int(p)] for p in preds.cpu().numpy()])

test_df["label"] = pred_labels



In [7]:
# Write submission
SUBMISSION_PATH.write_text(test_df.to_csv(index=False))
print(f"Submission written to {SUBMISSION_PATH.resolve()}")
print(f"Submission preview (first 5 rows):\n{test_df.head()}")

# Verify row count matches
expected_rows = test_df.shape[0]
with open(SUBMISSION_PATH, "r") as f:
    lines = f.readlines()
actual_rows = len(lines) - 1  # exclude header
print(f"Expected rows: {expected_rows}, rows in CSV (excluding header): {actual_rows}")
assert actual_rows == expected_rows, "Row count mismatch in submission file"

Submission written to /kaggle/working/submission.csv
Submission preview (first 5 rows):
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      3
3  1234555380.jpg      1
4  1234571117.jpg      3
Expected rows: 2676, rows in CSV (excluding header): 2676
